================================================================
FILE : train.py
WHAT THIS FILE DOES :
  Step 1 - Load train.csv and test.csv
  Step 2 - Show training data and test data on screen
  Step 3 - Train Random Forest (300 trees)
  Step 4 - Check training accuracy
  Step 5 - Save trained model
  Step 6 - Show full summary below

HOW TO RUN :  python train.py
RUN AFTER  :  preprocess.py
================================================================

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

In [ ]:
os.makedirs("results", exist_ok=True)

In [ ]:
print("=" * 65)
print("        MODEL TRAINING  —  Step 2 of 4")
print("=" * 65)
print("        Model : Random Forest  (300 Decision Trees)")
print("=" * 65)

================================================================
STEP 1 : LOAD TRAINING AND TEST DATA
================================================================
train.csv = 17,401 rows  → model LEARNS from this
test.csv  = 4,351  rows  → used in evaluate.py (not touched here)

In [ ]:
print("\n[STEP 1]  Loading train.csv and test.csv...")

In [ ]:
train_df = pd.read_csv("data/train.csv")
test_df  = pd.read_csv("data/test.csv")

In [ ]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

In [ ]:
X_test  = test_df.drop(columns=["label"])
y_test  = test_df["label"]

In [ ]:
print(f"         train.csv loaded  :  {train_df.shape[0]} rows  x  {train_df.shape[1]} columns")
print(f"         test.csv  loaded  :  {test_df.shape[0]} rows  x  {test_df.shape[1]} columns")

In [ ]:
# Save feature names — needed in evaluate.py and predict.py
feature_names = list(X_train.columns)
with open("results/feature_names.pkl", "wb") as f:
    pickle.dump(feature_names, f)

================================================================
STEP 2 : SHOW TRAINING DATA AND TEST DATA
================================================================

In [ ]:
print("\n" + "=" * 65)
print("  TRAINING DATA  —  Full View")
print("=" * 65)
print(f"  Shape : {train_df.shape[0]} rows  x  {train_df.shape[1]} columns")
print(f"  First 5 rows (showing first 8 feature columns + label):")
print()
cols_show = list(X_train.columns[:8]) + ["label"]
print(train_df[cols_show].head(5).to_string(index=True))
print()
print(f"  Last 5 rows:")
print(train_df[cols_show].tail(5).to_string(index=True))

In [ ]:
print()
print("  LABEL DISTRIBUTION in Training Data:")
print(f"    Benign    (0)  :  {(y_train==0).sum()}  files  ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"    Malicious (1)  :  {(y_train==1).sum()}  files  ({(y_train==1).sum()/len(y_train)*100:.1f}%)")
print(f"    Total          :  {len(y_train)}  files")

In [ ]:
print()
print("  FEATURE STATISTICS (Training Data — first 5 features):")
print(X_train.iloc[:, :5].describe().round(2).to_string())

In [ ]:
print("\n" + "=" * 65)
print("  TEST DATA  —  Full View")
print("=" * 65)
print(f"  Shape : {test_df.shape[0]} rows  x  {test_df.shape[1]} columns")
print(f"  First 5 rows (showing first 8 feature columns + label):")
print()
print(test_df[cols_show].head(5).to_string(index=True))
print()
print(f"  Last 5 rows:")
print(test_df[cols_show].tail(5).to_string(index=True))

In [ ]:
print()
print("  LABEL DISTRIBUTION in Test Data:")
print(f"    Benign    (0)  :  {(y_test==0).sum()}  files  ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"    Malicious (1)  :  {(y_test==1).sum()}  files  ({(y_test==1).sum()/len(y_test)*100:.1f}%)")
print(f"    Total          :  {len(y_test)}  files")

In [ ]:
print()
print("  NOTE: Test data is NOT used during training.")
print("        It is only used in evaluate.py to check accuracy.")

================================================================
STEP 3 : TRAIN RANDOM FOREST
================================================================
WHAT IS RANDOM FOREST?

  Builds 300 Decision Trees. Each tree:
    → Sees a RANDOM subset of training files
    → Uses a RANDOM subset of features at each split
    → Grows its own complete tree of yes/no questions
    → Votes: Benign (0) or Malicious (1)

  Final answer = MAJORITY VOTE of all 300 trees
  Example: 285 say Malicious, 15 say Benign → MALICIOUS

WHY 300 TREES AND NOT JUST 1?
  One tree overfits → memorizes training data → fails on new files
  300 diverse trees → individual mistakes cancel out → better accuracy

PARAMETERS EXPLAINED:

  n_estimators = 300
    → Build 300 trees total
    → More trees = more accurate but slower
    → 300 is the best balance for your dataset size

  max_depth = None
    → Each tree grows until every leaf is pure
    → No depth limit
    → The randomness (not depth) controls overfitting here

  class_weight = "balanced"
    → Both classes treated with equal importance
    → Prevents model from ignoring the minority class
    → Your data is 50/50 so this is a safety measure

  random_state = 42
    → Fixed random seed so you get same result every run
    → Without this, results change slightly each time
    → 42 is just a convention, any number works

  n_jobs = -1
    → Use ALL CPU cores on your computer
    → Trains all 300 trees in parallel = much faster

In [ ]:
print("\n" + "=" * 65)
print("[STEP 3]  Training Random Forest...")
print("=" * 65)
print()
print("  Model  : RandomForestClassifier")
print("  Trees  : 300")
print("  Type   : Supervised (uses labels to learn)")
print()
print("  HOW IT WORKS:")
print("  → Builds 300 Decision Trees")
print("  → Each tree sees different random data and features")
print("  → All 300 trees vote on every new file")
print("  → Majority vote = final prediction")
print()
print("  Training in progress... (1-3 minutes)")
print()

In [ ]:
rf = RandomForestClassifier(
    n_estimators  = 300,        # number of trees
    max_depth     = None,       # grow trees fully
    class_weight  = "balanced", # equal importance to both classes
    random_state  = 42,         # reproducible results
    n_jobs        = -1          # use all CPU cores
)

In [ ]:
rf.fit(X_train, y_train)   # ← THE MODEL LEARNS HERE

In [ ]:
print("  Training complete!")

================================================================
STEP 4 : CHECK TRAINING ACCURACY
================================================================
Training accuracy = how well model learned the training data
We expect ~100% because Random Forest can perfectly fit training data
This is fine — we check REAL accuracy on test data in evaluate.py

In [ ]:
print("\n[STEP 4]  Checking training accuracy...")

In [ ]:
train_predictions = rf.predict(X_train)
train_correct     = (train_predictions == y_train).sum()
train_accuracy    = train_correct / len(y_train) * 100

In [ ]:
print(f"\n  Training Accuracy  :  {train_accuracy:.2f}%")
print(f"  Correct predictions:  {train_correct} out of {len(y_train)}")
print(f"  Wrong predictions  :  {len(y_train) - train_correct}")
print()
print("  NOTE: 100% training accuracy is EXPECTED for Random Forest.")
print("        It shows the model learned the training data perfectly.")
print("        Real accuracy on UNSEEN files is checked in evaluate.py")

In [ ]:
# Show what model predicted vs actual for first 10 training files
print()
print("  PREDICTIONS vs ACTUAL  (first 10 training files):")
print("  " + "-" * 45)
print(f"  {'File':>4}  {'Actual':>10}  {'Predicted':>10}  {'Correct?':>8}")
print("  " + "-" * 45)
for i in range(10):
    actual    = "Malicious" if y_train.iloc[i] == 1 else "Benign"
    predicted = "Malicious" if train_predictions[i] == 1 else "Benign"
    correct   = "YES" if y_train.iloc[i] == train_predictions[i] else "NO"
    print(f"  {i+1:>4}  {actual:>10}  {predicted:>10}  {correct:>8}")
print("  " + "-" * 45)

================================================================
STEP 5 : SAVE TRAINED MODEL
================================================================
pickle.dump saves the trained model to a .pkl file
Think of it as FREEZING the model into a file
Later in evaluate.py and predict.py we UNFREEZE and use it

In [ ]:
print("\n[STEP 5]  Saving trained model...")

In [ ]:
with open("results/random_forest.pkl", "wb") as f:
    pickle.dump(rf, f)

In [ ]:
print(f"         Model saved  →  results/random_forest.pkl")
print(f"         Feature names saved  →  results/feature_names.pkl")

In [ ]:
# ================================================================
# FINAL SUMMARY — printed below the code
# ================================================================
print("\n" + "=" * 65)
print("  TRAINING COMPLETE — FULL SUMMARY")
print("=" * 65)
print()
print("  TRAINING DATA USED")
print(f"    Total files used   :  {len(X_train)}")
print(f"    Benign files       :  {(y_train==0).sum()}")
print(f"    Malicious files    :  {(y_train==1).sum()}")
print(f"    Features           :  {X_train.shape[1]}")
print()
print("  TEST DATA  (saved for evaluate.py)")
print(f"    Total files        :  {len(X_test)}")
print(f"    Benign files       :  {(y_test==0).sum()}")
print(f"    Malicious files    :  {(y_test==1).sum()}")
print()
print("  RANDOM FOREST MODEL")
print(f"    Trees built        :  {rf.n_estimators}")
print(f"    Max depth          :  None  (unlimited)")
print(f"    Class weight       :  balanced")
print(f"    Training accuracy  :  {train_accuracy:.2f}%")
print(f"    Correct / Total    :  {train_correct} / {len(y_train)}")
print()
print("  FILES SAVED")
print(f"    results/random_forest.pkl  ←  trained model")
print(f"    results/feature_names.pkl  ←  list of feature names")
print()
print("=" * 65)
print("  NEXT STEP  →  Run  evaluate.py")
print("=" * 65)